In [18]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from xgboost import XGBRegressor

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

In [19]:
from pathlib import Path

folder = Path("../output_csv")

input_file = folder / "engineered.csv"

df = pd.read_csv(input_file, low_memory=False)

df.head()

,ClosePrice,source_month,LivingArea,DaysOnMarket,LotSizeSquareFeet,YearBuilt,BathroomsTotalInteger,BedroomsTotal,GarageSpaces,Latitude,...,school_district_Williams Unified,school_district_Willits Unified,school_district_Willows Unified,school_district_Windsor Unified,school_district_Wiseburn Unified,school_district_Woodlake Unified,school_district_Woodland Joint Unified,school_district_Yosemite Unified,school_district_Yuba City Unified,school_district_Yucaipa-Calimesa Joint Unified
0,890000.0,202506,3000.0,181,9600.0,2021.0,3.0,3.0,2.0,34.264692,...,0,0,0,0,0,0,0,0,0,0
1,1876384.0,202506,1800.0,87,10400.0,1963.0,3.0,3.0,2.0,34.107983,...,0,0,0,0,0,0,0,0,0,0
2,4820000.0,202506,4270.0,0,22505.0,1980.0,6.0,6.0,3.0,37.567434,...,0,0,0,0,0,0,0,0,0,0
3,865000.0,202506,1442.0,0,4800.0,1985.0,2.0,3.0,2.0,33.906058,...,0,0,0,0,0,0,0,0,0,0
4,875000.0,202506,1086.0,0,5500.0,1953.0,1.0,3.0,4.0,37.705919,...,0,0,0,0,0,0,0,0,0,0


In [20]:
target = "ClosePrice"

feature_cols = [
    column
    for column in df.columns
    if column not in [
        target,
        "source_month"
    ]
]

print(
    "Number of features:",
    len(feature_cols)
)

Number of features: 392


In [21]:
target = "ClosePrice"

feature_cols = [
    column
    for column in df.columns
    if column not in [
        target,
        "source_month"
    ]
]

print(
    "Number of features:",
    len(feature_cols)
)

Number of features: 392


In [22]:
df["source_month"] = (
    df["source_month"]
    .astype(str)
    .astype(int)
)

months = sorted(
    df["source_month"].unique()
)

print(
    "Available months:",
    months
)

Available months: [202506, 202507, 202508, 202509, 202510, 202511, 202512, 202601, 202602, 202603, 202604, 202605, 202606]


In [23]:
test_month = months[-1]

print(
    "Test month:",
    test_month
)

Test month: 202606


In [24]:
X_window = 12

train_months = months[
    -(X_window + 1):-1
]

validation_month = train_months[-1]

fitting_months = train_months[:-1]

print(
    "Fitting months:",
    fitting_months
)

print(
    "Validation month:",
    validation_month
)

print(
    "Test month:",
    test_month
)

Fitting months: [202506, 202507, 202508, 202509, 202510, 202511, 202512, 202601, 202602, 202603, 202604]
Validation month: 202605
Test month: 202606


In [25]:
fitting_df = df[
    df["source_month"].isin(
        fitting_months
    )
].copy()

validation_df = df[
    df["source_month"]
    == validation_month
].copy()

full_train_df = df[
    df["source_month"].isin(
        train_months
    )
].copy()

test_df = df[
    df["source_month"]
    == test_month
].copy()

In [26]:
X_fit = fitting_df[
    feature_cols
].copy()

y_fit = fitting_df[
    target
].copy()


X_validation = validation_df[
    feature_cols
].copy()

y_validation = validation_df[
    target
].copy()


X_train = full_train_df[
    feature_cols
].copy()

y_train = full_train_df[
    target
].copy()


X_test = test_df[
    feature_cols
].copy()

y_test = test_df[
    target
].copy()

## Week 7 Baseline

The Week 7 12-month XGBoost model used:

- max_depth = 3
- learning_rate = 0.05
- n_estimators = 200
- subsample = 0.8
- colsample_bytree = 0.8

Its test R² was approximately 0.4843.

In [27]:
baseline_model = XGBRegressor(
    objective="reg:squarederror",
    max_depth=3,
    learning_rate=0.05,
    n_estimators=200,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

baseline_model.fit(
    X_train,
    y_train
)

baseline_predictions = (
    baseline_model.predict(
        X_test
    )
)

baseline_r2 = r2_score(
    y_test,
    baseline_predictions
)

baseline_mae = mean_absolute_error(
    y_test,
    baseline_predictions
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        baseline_predictions
    )
)

print(
    f"Baseline Test R²: "
    f"{baseline_r2:.4f}"
)

print(
    f"Baseline MAE: "
    f"${baseline_mae:,.2f}"
)

print(
    f"Baseline RMSE: "
    f"${baseline_rmse:,.2f}"
)

Baseline Test R²: 0.4843
Baseline MAE: $435,269.85
Baseline RMSE: $1,103,416.04


In [28]:
enhanced_parameters = [
    {
        "max_depth": 3,
        "learning_rate": 0.03,
        "n_estimators": 300,
        "min_child_weight": 1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0,
        "reg_lambda": 1
    },

    {
        "max_depth": 3,
        "learning_rate": 0.03,
        "n_estimators": 500,
        "min_child_weight": 1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0,
        "reg_lambda": 1
    },

    {
        "max_depth": 4,
        "learning_rate": 0.03,
        "n_estimators": 300,
        "min_child_weight": 1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0,
        "reg_lambda": 1
    },

    {
        "max_depth": 3,
        "learning_rate": 0.05,
        "n_estimators": 300,
        "min_child_weight": 3,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0,
        "reg_lambda": 1
    },

    {
        "max_depth": 3,
        "learning_rate": 0.05,
        "n_estimators": 300,
        "min_child_weight": 5,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0,
        "reg_lambda": 1
    },

    {
        "max_depth": 3,
        "learning_rate": 0.05,
        "n_estimators": 300,
        "min_child_weight": 3,
        "subsample": 0.7,
        "colsample_bytree": 0.8,
        "reg_alpha": 0,
        "reg_lambda": 1
    },

    {
        "max_depth": 3,
        "learning_rate": 0.05,
        "n_estimators": 300,
        "min_child_weight": 3,
        "subsample": 0.9,
        "colsample_bytree": 0.8,
        "reg_alpha": 0,
        "reg_lambda": 1
    },

    {
        "max_depth": 3,
        "learning_rate": 0.05,
        "n_estimators": 300,
        "min_child_weight": 3,
        "subsample": 0.8,
        "colsample_bytree": 0.7,
        "reg_alpha": 0,
        "reg_lambda": 1
    },

    {
        "max_depth": 3,
        "learning_rate": 0.05,
        "n_estimators": 300,
        "min_child_weight": 3,
        "subsample": 0.8,
        "colsample_bytree": 0.9,
        "reg_alpha": 0,
        "reg_lambda": 1
    },

    {
        "max_depth": 3,
        "learning_rate": 0.05,
        "n_estimators": 300,
        "min_child_weight": 3,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.1,
        "reg_lambda": 1
    },

    {
        "max_depth": 3,
        "learning_rate": 0.05,
        "n_estimators": 300,
        "min_child_weight": 3,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 1,
        "reg_lambda": 5
    },

    {
        "max_depth": 4,
        "learning_rate": 0.03,
        "n_estimators": 500,
        "min_child_weight": 3,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.1,
        "reg_lambda": 5
    }
]

print(
    "Number of experiments:",
    len(enhanced_parameters)
)

Number of experiments: 12


In [29]:
enhancement_tuning_results = []

for model_number, parameters in enumerate(
    enhanced_parameters,
    start=1
):

    model = XGBRegressor(
        objective="reg:squarederror",

        max_depth=parameters[
            "max_depth"
        ],

        learning_rate=parameters[
            "learning_rate"
        ],

        n_estimators=parameters[
            "n_estimators"
        ],

        min_child_weight=parameters[
            "min_child_weight"
        ],

        subsample=parameters[
            "subsample"
        ],

        colsample_bytree=parameters[
            "colsample_bytree"
        ],

        reg_alpha=parameters[
            "reg_alpha"
        ],

        reg_lambda=parameters[
            "reg_lambda"
        ],

        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )


    model.fit(
        X_fit,
        y_fit
    )


    validation_predictions = model.predict(
        X_validation
    )


    validation_r2 = r2_score(
        y_validation,
        validation_predictions
    )

    validation_mae = mean_absolute_error(
        y_validation,
        validation_predictions
    )

    validation_rmse = np.sqrt(
        mean_squared_error(
            y_validation,
            validation_predictions
        )
    )


    enhancement_tuning_results.append(
        {
            "experiment": model_number,

            **parameters,

            "validation_r2": validation_r2,

            "validation_mae": validation_mae,

            "validation_rmse": validation_rmse
        }
    )

In [30]:
enhancement_tuning_df = pd.DataFrame(
    enhancement_tuning_results
)

enhancement_tuning_df = (
    enhancement_tuning_df
    .sort_values(
        by="validation_rmse",
        ascending=True
    )
    .reset_index(
        drop=True
    )
)

enhancement_tuning_df

,experiment,max_depth,learning_rate,n_estimators,min_child_weight,subsample,colsample_bytree,reg_alpha,reg_lambda,validation_r2,validation_mae,validation_rmse
0,7,3,0.05,300,3,0.9,0.8,0.0,1,0.155375,473948.619703,1.542857e+06
1,11,3,0.05,300,3,0.8,0.8,1.0,5,0.106770,497543.182136,1.586629e+06
2,8,3,0.05,300,3,0.8,0.7,0.0,1,0.082006,498716.590565,1.608473e+06
3,5,3,0.05,300,5,0.8,0.8,0.0,1,-0.059531,514152.840962,1.728028e+06
4,4,3,0.05,300,3,0.8,0.8,0.0,1,-0.086687,499452.685816,1.750033e+06
5,10,3,0.05,300,3,0.8,0.8,0.1,1,-0.086687,499452.687001,1.750033e+06
6,6,3,0.05,300,3,0.7,0.8,0.0,1,-0.097661,514922.518868,1.758847e+06
7,9,3,0.05,300,3,0.8,0.9,0.0,1,-0.139463,501825.628649,1.792025e+06
8,12,4,0.03,500,3,0.8,0.8,0.1,5,-0.226674,473285.085058,1.859338e+06
9,1,3,0.03,300,1,0.8,0.8,0.0,1,-1.324368,486962.227931,2.559448e+06


In [31]:
enhancement_tuning_df.head(
    10
)

,experiment,max_depth,learning_rate,n_estimators,min_child_weight,subsample,colsample_bytree,reg_alpha,reg_lambda,validation_r2,validation_mae,validation_rmse
0,7,3,0.05,300,3,0.9,0.8,0.0,1,0.155375,473948.619703,1.542857e+06
1,11,3,0.05,300,3,0.8,0.8,1.0,5,0.106770,497543.182136,1.586629e+06
2,8,3,0.05,300,3,0.8,0.7,0.0,1,0.082006,498716.590565,1.608473e+06
3,5,3,0.05,300,5,0.8,0.8,0.0,1,-0.059531,514152.840962,1.728028e+06
4,4,3,0.05,300,3,0.8,0.8,0.0,1,-0.086687,499452.685816,1.750033e+06
5,10,3,0.05,300,3,0.8,0.8,0.1,1,-0.086687,499452.687001,1.750033e+06
6,6,3,0.05,300,3,0.7,0.8,0.0,1,-0.097661,514922.518868,1.758847e+06
7,9,3,0.05,300,3,0.8,0.9,0.0,1,-0.139463,501825.628649,1.792025e+06
8,12,4,0.03,500,3,0.8,0.8,0.1,5,-0.226674,473285.085058,1.859338e+06
9,1,3,0.03,300,1,0.8,0.8,0.0,1,-1.324368,486962.227931,2.559448e+06


In [32]:
best_parameters = (
    enhancement_tuning_df
    .iloc[0]
)

best_parameters

experiment          7.000000e+00
max_depth           3.000000e+00
learning_rate       5.000000e-02
n_estimators        3.000000e+02
min_child_weight    3.000000e+00
subsample           9.000000e-01
colsample_bytree    8.000000e-01
reg_alpha           0.000000e+00
reg_lambda          1.000000e+00
validation_r2       1.553755e-01
validation_mae      4.739486e+05
validation_rmse     1.542857e+06
Name: 0, dtype: float64

In [33]:
print(
    "Selected parameters:"
)

print(
    "max_depth:",
    best_parameters[
        "max_depth"
    ]
)

print(
    "learning_rate:",
    best_parameters[
        "learning_rate"
    ]
)

print(
    "n_estimators:",
    best_parameters[
        "n_estimators"
    ]
)

print(
    "min_child_weight:",
    best_parameters[
        "min_child_weight"
    ]
)

print(
    "subsample:",
    best_parameters[
        "subsample"
    ]
)

print(
    "colsample_bytree:",
    best_parameters[
        "colsample_bytree"
    ]
)

print(
    "reg_alpha:",
    best_parameters[
        "reg_alpha"
    ]
)

print(
    "reg_lambda:",
    best_parameters[
        "reg_lambda"
    ]
)

Selected parameters:
max_depth: 3.0
learning_rate: 0.05
n_estimators: 300.0
min_child_weight: 3.0
subsample: 0.9
colsample_bytree: 0.8
reg_alpha: 0.0
reg_lambda: 1.0


In [34]:
enhanced_model = XGBRegressor(
    objective="reg:squarederror",

    max_depth=int(
        best_parameters[
            "max_depth"
        ]
    ),

    learning_rate=float(
        best_parameters[
            "learning_rate"
        ]
    ),

    n_estimators=int(
        best_parameters[
            "n_estimators"
        ]
    ),

    min_child_weight=float(
        best_parameters[
            "min_child_weight"
        ]
    ),

    subsample=float(
        best_parameters[
            "subsample"
        ]
    ),

    colsample_bytree=float(
        best_parameters[
            "colsample_bytree"
        ]
    ),

    reg_alpha=float(
        best_parameters[
            "reg_alpha"
        ]
    ),

    reg_lambda=float(
        best_parameters[
            "reg_lambda"
        ]
    ),

    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

In [36]:
enhanced_model.fit(
    X_train,
    y_train
)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [37]:
enhanced_train_predictions = (
    enhanced_model.predict(
        X_train
    )
)

enhanced_test_predictions = (
    enhanced_model.predict(
        X_test
    )
)

In [38]:
enhanced_train_r2 = r2_score(
    y_train,
    enhanced_train_predictions
)

enhanced_test_r2 = r2_score(
    y_test,
    enhanced_test_predictions
)

enhanced_mae = mean_absolute_error(
    y_test,
    enhanced_test_predictions
)

enhanced_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        enhanced_test_predictions
    )
)

print(
    f"Train R²: "
    f"{enhanced_train_r2:.4f}"
)

print(
    f"Test R²: "
    f"{enhanced_test_r2:.4f}"
)

print(
    f"MAE: "
    f"${enhanced_mae:,.2f}"
)

print(
    f"RMSE: "
    f"${enhanced_rmse:,.2f}"
)

Train R²: 0.1533
Test R²: 0.5544
MAE: $447,720.20
RMSE: $1,025,668.90


In [39]:
comparison_df = pd.DataFrame(
    {
        "model": [
            "Week 7 Baseline",
            "Week 9 Enhanced"
        ],

        "test_r2": [
            baseline_r2,
            enhanced_test_r2
        ],

        "mae": [
            baseline_mae,
            enhanced_mae
        ],

        "rmse": [
            baseline_rmse,
            enhanced_rmse
        ]
    }
)

comparison_df

,model,test_r2,mae,rmse
0,Week 7 Baseline,0.484318,435269.853813,1.103416e+06
1,Week 9 Enhanced,0.554428,447720.196187,1.025669e+06


In [41]:
r2_improvement = (
    enhanced_test_r2
    - baseline_r2
)

print(
    f"Baseline Test R²: "
    f"{baseline_r2:.4f}"
)

print(
    f"Enhanced Test R²: "
    f"{enhanced_test_r2:.4f}"
)

print(
    f"R² Improvement: "
    f"{r2_improvement:+.4f}"
)

Baseline Test R²: 0.4843
Enhanced Test R²: 0.5544
R² Improvement: +0.0701


In [42]:
enhanced_model_results_df = pd.DataFrame(
    [
        {
            "model": "Week 7 Baseline",
            "training_window_months": 12,

            "max_depth": 3,
            "learning_rate": 0.05,
            "n_estimators": 200,
            "min_child_weight": 1,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "reg_alpha": 0,
            "reg_lambda": 1,

            "test_r2": baseline_r2,
            "mae": baseline_mae,
            "rmse": baseline_rmse
        },

        {
            "model": "Week 9 Enhanced",
            "training_window_months": 12,

            "max_depth": int(
                best_parameters[
                    "max_depth"
                ]
            ),

            "learning_rate": float(
                best_parameters[
                    "learning_rate"
                ]
            ),

            "n_estimators": int(
                best_parameters[
                    "n_estimators"
                ]
            ),

            "min_child_weight": float(
                best_parameters[
                    "min_child_weight"
                ]
            ),

            "subsample": float(
                best_parameters[
                    "subsample"
                ]
            ),

            "colsample_bytree": float(
                best_parameters[
                    "colsample_bytree"
                ]
            ),

            "reg_alpha": float(
                best_parameters[
                    "reg_alpha"
                ]
            ),

            "reg_lambda": float(
                best_parameters[
                    "reg_lambda"
                ]
            ),

            "test_r2": enhanced_test_r2,
            "mae": enhanced_mae,
            "rmse": enhanced_rmse
        }
    ]
)

enhanced_model_results_df

,model,training_window_months,max_depth,learning_rate,n_estimators,min_child_weight,subsample,colsample_bytree,reg_alpha,reg_lambda,test_r2,mae,rmse
0,Week 7 Baseline,12,3,0.05,200,1.0,0.8,0.8,0.0,1.0,0.484318,435269.853813,1.103416e+06
1,Week 9 Enhanced,12,3,0.05,300,3.0,0.9,0.8,0.0,1.0,0.554428,447720.196187,1.025669e+06


In [43]:
enhanced_model_results_df.to_csv(
    folder
    / "enhanced_model_results.csv",
    index=False
)

print(
    "Saved:",
    folder
    / "enhanced_model_results.csv"
)

Saved: ..\output_csv\enhanced_model_results.csv




The Week 7 12-month XGBoost model was used as the baseline because it
had the highest test R² among the original training windows.

For Week 9, additional XGBoost hyperparameters were explored, including
minimum child weight, row subsampling, feature subsampling, and L1/L2
regularization. Model s